# Baseline GPT-2 cho Vietnamese Math Word Problems

Notebook này fine-tune `NlpHUST/gpt2-vietnamese` để sinh **cả lời giải toán tiếng Việt**, rồi kết thúc bằng câu đáp án cuối:

```text
Đáp án là: <số>
```

`model_output` khi dự đoán sẽ giống `sample_prediction.json`: một đoạn giải ngắn, không phải chỉ có mỗi câu đáp án.

Bản này thêm data processing dựa trên `eda_train.ipynb`: kiểm tra thiếu trường, chuẩn hóa text, trích đáp án, chuẩn hóa target và bỏ trùng cặp `query_vi + response_vi` trước khi train.

## Run

Kaggle: GPU ON, Internet OFF.

Notebook chỉ ghi các file cần thiết:

- `gpt2_math_baseline_ckpt/`: checkpoint sau fine-tune.
- `valid_output.json`: output validation để tự kiểm tra.
- `test_predictions.json`: file nộp nếu có `test.json`.


In [1]:
# 1. Import và kiểm tra môi trường
import os
import sys
import gc
import re
import json
import math
import time
import random
import hashlib
import inspect
from pathlib import Path
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

try:
    from IPython.display import display
except Exception:
    display = print

try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

pd.set_option("display.max_colwidth", 180)

print("Python:", sys.version.replace("\n", " "))
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
CUDA_OK = torch.cuda.is_available()
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        capability = torch.cuda.get_device_capability(i)
        print(f"GPU {i}:", name, "| capability:", capability)
    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        CUDA_OK = False
        print("WARNING: GPU hiện tại có compute capability < 7.0, không tương thích với PyTorch CUDA trong log Kaggle này.")
        print("Nếu muốn train trên Kaggle, hãy chọn GPU T4/V100/A100 thay vì P100. Notebook sẽ không train full được trên GPU này.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
GPU 0: Tesla T4 | capability: (7, 5)
GPU 1: Tesla T4 | capability: (7, 5)


In [2]:
# 2. Đường dẫn dữ liệu, model và output
# Nếu Kaggle mount input khác tên, chỉ cần sửa DATA_DIR hoặc MODEL_DIR ở đây.
def first_existing(*paths):
    checked = []
    for p in paths:
        p = Path(p)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào:\n" + "\n".join(checked))


def first_existing_optional(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    PROJECT_ROOT / "data",
)

MODEL_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/nlphust-gpt2-vietnamese",
    PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese",
)

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else PROJECT_ROOT / "outputs" / "baseline_gpt2_math"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"
TEST_FILE = first_existing_optional(DATA_DIR / "test.json", "/kaggle/input/test.json")

OUTPUT_DIR = WORK_DIR / "gpt2_math_baseline_ckpt"
VALID_OUTPUT_PATH = WORK_DIR / "valid_output.json"
TEST_OUTPUT_PATH = WORK_DIR / "test_predictions.json"

SAFE_EOS_ID = 50256
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("WORK_DIR:", WORK_DIR)
print("TEST_FILE:", TEST_FILE)

DATA_DIR: /kaggle/input/datasets/kimanh2002/dataset-math
MODEL_DIR: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
WORK_DIR: /kaggle/working
TEST_FILE: None


In [3]:
# 3. Đọc dữ liệu
# Muốn chạy thử nhanh thì đổi các limit bên dưới thành số nhỏ, ví dụ 2000 và 100.
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None


def load_records(path, need_response=False):
    path = Path(path)
    with path.open("r", encoding="utf-8-sig") as f:
        first = f.read(1)
        f.seek(0)
        records = json.load(f) if first == "[" else [json.loads(line) for line in f if line.strip()]

    out = []
    for i, rec in enumerate(records):
        if "query_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu query_vi")
        if need_response and "response_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu response_vi")
        item = dict(rec)
        item.setdefault("id", i)
        item.setdefault("type", "unknown")
        out.append(item)
    return out


raw_train = load_records(TRAIN_FILE, need_response=True)
raw_valid = load_records(VALID_FILE, need_response=True) if VALID_FILE.exists() else []
raw_test = load_records(TEST_FILE) if TEST_FILE else []

if MAX_TRAIN_SAMPLES is not None:
    raw_train = raw_train[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES is not None:
    raw_valid = raw_valid[:MAX_VALID_SAMPLES]
if MAX_TEST_SAMPLES is not None:
    raw_test = raw_test[:MAX_TEST_SAMPLES]

print("raw train:", len(raw_train))
print("raw valid:", len(raw_valid))
print("raw test :", len(raw_test))
print(json.dumps(raw_train[0], ensure_ascii=False, indent=2)[:1400])

raw train: 100000
raw valid: 1000
raw test : 0
{
  "original_question_vi": "Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn chuẩn bị thêm mười đĩa đề phòng trường hợp có sự cố xảy ra. Mỗi đĩa bít tết và măng tây sốt bơ tỏi sẽ có 8 ngọn măng tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?",
  "original_question_en": "Bridgette and Alex are getting married. Bridgette is inviting 84 guests, and Alex is inviting two thirds of that number of guests. They hired a caterer to make a plated meal for each guest at the wedding reception. The caterer always makes ten extra plates just in case something goes wrong. Each plate of steak and asparagus in garlic butter will have 8 asparagus spears on it. How many asparagus spears will the caterer need in all?",
  "query_vi": "Bridgette và Alex sắp kết hôn. Bridget

In [4]:
# 4. Hàm trích đáp án và tính điểm
# Dùng cho cả data processing và validation.
ANSWER_ANCHORS = [
    r"Đáp án là\s*[:：]?",
    r"Câu trả lời là\s*[:：]?",
    r"(?:Câu\s+)?Trả lời(?:\s+là)?\s*[:：]?",
    r"Đáp án\s*[:：]?",
    r"The answer is\s*[:：]?",
    r"Answer\s*[:：]?",
    r"####\s*",
]
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
NUM_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")


def clean_answer_tail(text):
    if text is None:
        return None
    text = str(text).strip().split("\n", 1)[0]
    text = re.sub(r"^(?:là|=|:|：)\s*", "", text, flags=re.IGNORECASE)
    text = text.strip(" .。;；,，")
    return text or None


def extract_answer_text(text, allow_last_number=False):
    text = str(text or "")
    matches = []
    for pattern in ANSWER_ANCHORS:
        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            matches.append((m.start(), m.end(), text[m.end():]))
    if matches:
        _, _, tail = sorted(matches, key=lambda x: (x[0], x[1]))[-1]
        return clean_answer_tail(tail)

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    if allow_last_number:
        nums = NUM_RE.findall(text)
        if nums:
            return clean_answer_tail(nums[-1])
    return None


def parse_plain_number(text):
    text = str(text).strip().replace(" ", "")
    if not text:
        return None

    if "/" in text:
        parts = text.split("/")
        if len(parts) == 2:
            a = parse_plain_number(parts[0])
            b = parse_plain_number(parts[1])
            if a is not None and b not in (None, 0):
                return a / b
        return None

    if "," in text and "." not in text:
        right = text.split(",")[-1]
        text = text.replace(",", "") if len(right) == 3 else text.replace(",", ".")
    elif "," in text and "." in text:
        text = text.replace(",", "")

    try:
        out = float(text)
    except ValueError:
        return None
    return out if math.isfinite(out) else None


def parse_number(text):
    if text is None:
        return None

    text = str(text).strip()
    if not text:
        return None

    direct = parse_plain_number(text)
    if direct is not None:
        return direct

    m = NUM_RE.search(text)
    return parse_plain_number(m.group(0)) if m else None


def relative_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))


def score_one(rel_err, extractable=True):
    if not extractable or rel_err is None:
        return 0
    if rel_err <= 0.01:
        return 10
    if rel_err <= 0.10:
        return 5
    if rel_err <= 0.50:
        return 1
    return 0

In [5]:
# 5. Data processing trước khi train
# Các tham số ở cell này chỉ liên quan đến xử lý dữ liệu.
DROP_DUPLICATE_PAIRS = True       # EDA cho thấy có cặp query-response trùng.
DROP_TRAIN_WITHOUT_ANSWER = False # Bật True nếu muốn bỏ mẫu không tách được đáp án số.
MAX_QUERY_WORDS = None            # Ví dụ: 180 nếu muốn bỏ câu hỏi quá dài.
MAX_RESPONSE_WORDS = None         # Ví dụ: 320 nếu muốn bỏ lời giải quá dài.


def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def word_count(text):
    return len(re.findall(r"\S+", str(text or "")))


def normalized_hash(text):
    text = normalize_space(text).lower()
    return hashlib.blake2b(text.encode("utf-8"), digest_size=16).hexdigest()


def normalize_response(response):
    text = normalize_space(response)
    answer = extract_answer_text(text, allow_last_number=True)

    text = re.sub(r"\s*####\s*[-+]?\d[\d,./]*", "", text).strip()
    text = re.sub(r"Câu trả lời là\s*[:：]?", "Đáp án là:", text, flags=re.IGNORECASE)
    text = re.sub(r"The answer is\s*[:：]?", "Đáp án là:", text, flags=re.IGNORECASE)
    text = re.sub(r"(?:Câu\s+)?Trả lời(?:\s+là)?\s*[:：]?", "Đáp án là:", text, flags=re.IGNORECASE)

    if answer and not re.search(r"Đáp án là\s*[:：]?", text[-180:], flags=re.IGNORECASE):
        text = text.rstrip(" .。;；,，") + f". Đáp án là: {answer}"
    return normalize_space(text)


def process_train(records):
    kept = []
    drop_reasons = []
    seen_pairs = set()

    for i, rec in enumerate(records):
        query = normalize_space(rec.get("query_vi"))
        response = normalize_response(rec.get("response_vi"))
        answer_text = extract_answer_text(response, allow_last_number=True)
        answer_num = parse_number(answer_text)
        q_words = word_count(query)
        r_words = word_count(response)

        reason = None
        if not query or not response:
            reason = "missing_query_or_response"
        elif DROP_TRAIN_WITHOUT_ANSWER and answer_num is None:
            reason = "no_numeric_answer"
        elif MAX_QUERY_WORDS is not None and q_words > MAX_QUERY_WORDS:
            reason = "query_too_long"
        elif MAX_RESPONSE_WORDS is not None and r_words > MAX_RESPONSE_WORDS:
            reason = "response_too_long"
        else:
            pair_hash = normalized_hash(query + "\n" + response)
            if DROP_DUPLICATE_PAIRS and pair_hash in seen_pairs:
                reason = "duplicate_pair"
            seen_pairs.add(pair_hash)

        if reason:
            drop_reasons.append(reason)
            continue

        item = dict(rec)
        item["id"] = rec.get("id", i)
        item["query_vi"] = query
        item["response_vi_raw"] = rec.get("response_vi")
        item["response_vi"] = response
        item["answer_text"] = answer_text
        item["answer_num"] = answer_num
        item["query_words"] = q_words
        item["response_words"] = r_words
        kept.append(item)

    return kept, Counter(drop_reasons)


def process_eval_or_test(records, has_response):
    out = []
    for i, rec in enumerate(records):
        item = dict(rec)
        item["id"] = rec.get("id", i)
        item["query_vi"] = normalize_space(rec.get("query_vi"))
        item.setdefault("type", "unknown")
        if has_response:
            item["response_vi_raw"] = rec.get("response_vi")
            item["response_vi"] = normalize_response(rec.get("response_vi"))
            item["answer_text"] = extract_answer_text(item["response_vi"], allow_last_number=True)
            item["answer_num"] = parse_number(item["answer_text"])
        out.append(item)
    return out


train_records, drop_counter = process_train(raw_train)
valid_records = process_eval_or_test(raw_valid, has_response=True)
test_records = process_eval_or_test(raw_test, has_response=False)

print("train before:", len(raw_train), "| after:", len(train_records), "| dropped:", sum(drop_counter.values()))
print("drop reasons:", dict(drop_counter))
print("valid:", len(valid_records), "| test:", len(test_records))
print("\nTarget sau xử lý:")
print(train_records[0]["response_vi"][:1000])

train before: 100000 | after: 98935 | dropped: 1065
drop reasons: {'duplicate_pair': 1065}
valid: 1000 | test: 0

Target sau xử lý:
Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó, tức là 84 * 2/3 = 56 khách. Vậy tổng số khách là 84 + 56 = 140 khách. Người phục vụ luôn làm thêm 10 đĩa nên tổng số đĩa cần dùng là 140 + 10 = 150 đĩa. Mỗi đĩa sẽ có 8 ngọn măng tây trên đó, như vậy tổng số ngọn măng tây cần thiết là 150 * 8 = 1200 ngọn măng tây. Đáp án là: 1200


In [6]:
# 6. Kiểm tra dữ liệu sau processing
# Phần này lấy ý từ eda_train.ipynb: type, độ dài, đáp án và trùng lặp.
def feature_df(records, split):
    rows = []
    for i, rec in enumerate(records):
        rows.append({
            "split": split,
            "index": i,
            "type": rec.get("type", "unknown"),
            "query_words": word_count(rec.get("query_vi")),
            "response_words": word_count(rec.get("response_vi", "")),
            "has_answer": rec.get("answer_num") is not None,
            "answer_num": rec.get("answer_num"),
            "pair_hash": normalized_hash(rec.get("query_vi", "") + "\n" + rec.get("response_vi", "")),
        })
    return pd.DataFrame(rows)


train_df = feature_df(train_records, "train")
valid_df = feature_df(valid_records, "valid") if valid_records else pd.DataFrame()

print("Phân bố type sau processing:")
display(train_df["type"].value_counts().rename_axis("type").reset_index(name="count"))

print("Độ dài train:")
display(train_df[["query_words", "response_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))

print("Độ dài và answer rate theo type:")
by_type = (
    train_df.groupby("type")
    .agg(
        count=("index", "count"),
        query_p95=("query_words", lambda s: s.quantile(0.95)),
        response_p95=("response_words", lambda s: s.quantile(0.95)),
        answer_rate=("has_answer", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)
display(by_type.round(3))

print("Tỷ lệ có đáp án số:", round(float(train_df["has_answer"].mean()), 4))
print("Số cặp query-response còn trùng:", int(train_df["pair_hash"].duplicated().sum()))

Phân bố type sau processing:


,type,count
0,GSM_Rephrased,20166
1,GSM_AnsAug,19824
2,MATH_AnsAug,18735
3,MATH_Rephrased,12744
4,GSM_FOBAR,10141
5,GSM_SV,9891
6,MATH_FOBAR,3747
7,MATH_SV,3687


Độ dài train:


,query_words,response_words
count,98935.00,98935.00
mean,48.67,112.90
std,26.95,65.92
min,2.00,3.00
50%,46.00,96.00
90%,83.00,199.00
95%,96.00,236.00
99%,126.00,335.00
max,311.00,771.00


Độ dài và answer rate theo type:


,type,count,query_p95,response_p95,answer_rate
2,GSM_Rephrased,20166,80.0,154.0,1.000
0,GSM_AnsAug,19824,92.0,152.0,1.000
4,MATH_AnsAug,18735,67.0,171.0,0.990
6,MATH_Rephrased,12744,58.0,182.0,0.993
1,GSM_FOBAR,10141,120.0,235.0,1.000
3,GSM_SV,9891,107.0,279.0,1.000
5,MATH_FOBAR,3747,117.0,429.0,1.000
7,MATH_SV,3687,104.0,357.0,1.000


Tỷ lệ có đáp án số: 0.9971
Số cặp query-response còn trùng: 0


In [7]:
# 7. Prompt, tokenizer và audit token length
# MAX_LENGTH là thông số liên quan trực tiếp đến tokenizer/dataset nên đặt ở đây.
MAX_LENGTH = 512
TOKEN_AUDIT_SAMPLES = 2000
PROMPT_TEMPLATE = (
    "Bạn là trợ lý giải toán. Hãy viết lời giải ngắn gọn bằng tiếng Việt, rồi kết thúc bằng đúng dạng 'Đáp án là: <số>'.\n"
    "Câu hỏi: {q}\n"
    "Lời giải:"
)


def build_prompt(rec):
    return PROMPT_TEMPLATE.format(q=rec["query_vi"])


tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

print("Tokenizer vocab_size:", getattr(tokenizer, "vocab_size", None), "| len:", len(tokenizer))
print("pad_token_id:", tokenizer.pad_token_id, "| eos_token_id:", tokenizer.eos_token_id)

sample = train_records if len(train_records) <= TOKEN_AUDIT_SAMPLES else random.sample(train_records, TOKEN_AUDIT_SAMPLES)
token_rows = []
for rec in tqdm(sample, desc="token audit"):
    p_ids = tokenizer(build_prompt(rec), add_special_tokens=False)["input_ids"]
    r_ids = tokenizer(rec["response_vi"], add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]
    token_rows.append({
        "type": rec.get("type"),
        "prompt_tokens": len(p_ids),
        "response_tokens": len(r_ids),
        "total_tokens": len(p_ids) + len(r_ids),
        "will_truncate": len(p_ids) + len(r_ids) > MAX_LENGTH,
    })

token_df = pd.DataFrame(token_rows)
display(token_df[["prompt_tokens", "response_tokens", "total_tokens"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
print("Ước lượng tỷ lệ bị cắt:", round(float(token_df["will_truncate"].mean()), 4))
display(token_df.groupby("type")["will_truncate"].mean().sort_values(ascending=False).to_frame("truncate_rate"))

Tokenizer vocab_size: 50257 | len: 50258
pad_token_id: 50256 | eos_token_id: 50256


token audit:   0%|          | 0/2000 [00:00<?, ?it/s]

,prompt_tokens,response_tokens,total_tokens
count,2000.00,2000.00,2000.00
mean,99.96,155.62,255.58
std,37.69,94.51,113.22
min,46.00,11.00,83.00
50%,94.00,133.50,229.50
90%,136.00,270.00,393.00
95%,156.05,330.00,462.00
99%,221.02,531.05,684.00
max,701.00,857.00,1045.00


Ước lượng tỷ lệ bị cắt: 0.0355


,truncate_rate
type,
MATH_FOBAR,0.370370
MATH_SV,0.123457
MATH_Rephrased,0.045775
MATH_AnsAug,0.027624
GSM_FOBAR,0.023256
GSM_SV,0.011494
GSM_AnsAug,0.002506
GSM_Rephrased,0.000000


In [8]:
# 8. Dataset cho supervised fine-tuning
# Loss chỉ tính trên phần lời giải, không tính trên prompt.
def clamp_ids(ids, vocab_size):
    return [min(max(int(x), 0), vocab_size - 1) for x in ids]


def fit_prompt_response(prompt_ids, response_ids, max_length):
    if len(prompt_ids) + len(response_ids) <= max_length:
        return prompt_ids, response_ids

    prompt_room = min(len(prompt_ids), max(32, int(max_length * 0.55)))
    prompt_ids = prompt_ids[:prompt_room]
    room = max_length - len(prompt_ids)

    if len(response_ids) > room:
        tail = min(64, max(16, room // 4))
        head = max(0, room - tail)
        response_ids = response_ids[:head] + response_ids[-tail:]
    return prompt_ids, response_ids[: max_length - len(prompt_ids)]


class MathDataset(Dataset):
    def __init__(self, records, tokenizer, vocab_size, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.vocab_size = vocab_size
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt_ids = self.tokenizer(build_prompt(rec), add_special_tokens=False)["input_ids"]
        response_ids = self.tokenizer(rec["response_vi"], add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]
        prompt_ids, response_ids = fit_prompt_response(prompt_ids, response_ids, self.max_length)

        input_ids = clamp_ids(prompt_ids + response_ids, self.vocab_size)
        labels = [-100] * len(prompt_ids) + clamp_ids(response_ids, self.vocab_size)
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        max_len = int(math.ceil(max_len / 8) * 8)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in batch:
            pad = max_len - len(item["input_ids"])
            out["input_ids"].append(item["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(item["attention_mask"] + [0] * pad)
            out["labels"].append(item["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

In [9]:
# 9. Tham số train và tạo Trainer
# Đây là các thông số thường đổi khi train.
RUN_TRAIN = True
EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM = 8
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.03
LOGGING_STEPS = 20
TRAINER_EVAL_SAMPLES = 1000

tmp_model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
tmp_model.config.pad_token_id = SAFE_EOS_ID
tmp_model.config.eos_token_id = SAFE_EOS_ID
MODEL_VOCAB_SIZE = tmp_model.get_input_embeddings().num_embeddings
del tmp_model
gc.collect()
torch.cuda.empty_cache()

train_ds = MathDataset(train_records, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH)
eval_records_for_trainer = valid_records[:TRAINER_EVAL_SAMPLES]
eval_ds = MathDataset(eval_records_for_trainer, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH) if eval_records_for_trainer else None
collator = PadCollator()

effective_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count() if CUDA_OK else 0)
print("train samples:", len(train_ds), "| eval samples:", len(eval_ds) if eval_ds else 0)
print("effective batch:", effective_batch)
print("steps/epoch:", math.ceil(len(train_ds) / effective_batch))


def make_training_args():
    use_bf16 = bool(CUDA_OK and torch.cuda.is_bf16_supported())
    use_fp16 = bool(CUDA_OK and not use_bf16)
    kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        logging_steps=LOGGING_STEPS,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=2 if IS_KAGGLE else 0,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch" if eval_ds else "no"
    else:
        kwargs["evaluation_strategy"] = "epoch" if eval_ds else "no"
    if "bf16" in sig.parameters:
        kwargs["bf16"] = use_bf16
    if "fp16" in sig.parameters:
        kwargs["fp16"] = use_fp16
    if not CUDA_OK:
        if "use_cpu" in sig.parameters:
            kwargs["use_cpu"] = True
        elif "no_cuda" in sig.parameters:
            kwargs["no_cuda"] = True
    return TrainingArguments(**kwargs)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train samples: 98935 | eval samples: 1000
effective batch: 64
steps/epoch: 1546


In [10]:
# 10. Train và lưu checkpoint
if RUN_TRAIN:
    if not CUDA_OK:
        raise RuntimeError(
            "Không có GPU CUDA dùng được cho full training. Nếu Kaggle đang cấp Tesla P100 như log, "
            "hãy đổi Accelerator sang T4/V100/A100. Nếu chỉ muốn chạy thử notebook, đặt RUN_TRAIN=False "
            "hoặc giảm MAX_TRAIN_SAMPLES xuống rất nhỏ."
        )
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    trainer = Trainer(
        model=model,
        args=make_training_args(),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
    )

    start = time.time()
    train_output = trainer.train()
    print("Train minutes:", round((time.time() - start) / 60, 2))
    print(train_output)

    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skip train. Inference sẽ dùng checkpoint nếu có, nếu không dùng base model.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.232834,1.212889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train minutes: 135.23
TrainOutput(global_step=1546, training_loss=1.3488761348526968, metrics={'train_runtime': 8113.756, 'train_samples_per_second': 12.193, 'train_steps_per_second': 0.191, 'total_flos': 2.095604964864e+16, 'train_loss': 1.3488761348526968, 'epoch': 1.0})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
# 11. Hàm sinh lời giải
# Các thông số generation đặt ngay tại đây để dễ chỉnh khi test.
MAX_NEW_TOKENS = 192
NUM_BEAMS = 1
DO_SAMPLE = False
REPETITION_PENALTY = 1.05
NO_REPEAT_NGRAM_SIZE = 4


def save_json(obj, path):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def postprocess_output(text):
    text = str(text).strip()
    for marker in ["\nCâu hỏi:", "\nQuestion:", "\n###"]:
        pos = text.find(marker)
        if pos >= 0:
            text = text[:pos].strip()
    return text


def generate_predictions(model_dir, records, output_path, name):
    device = "cuda" if CUDA_OK else "cpu"
    print("Load for generation:", model_dir, "| device:", device)

    gen_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
    gen_tokenizer.pad_token_id = SAFE_EOS_ID
    gen_tokenizer.eos_token_id = SAFE_EOS_ID

    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(str(model_dir), torch_dtype=dtype, local_files_only=True).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()

    vocab_size = model.get_input_embeddings().num_embeddings
    outputs = []
    start_all = time.time()

    with torch.inference_mode():
        for i, rec in enumerate(tqdm(records, desc=name)):
            enc = gen_tokenizer(build_prompt(rec), return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
            input_ids = enc["input_ids"].clamp(max=vocab_size - 1)
            gen = model.generate(
                input_ids=input_ids,
                attention_mask=enc.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=SAFE_EOS_ID,
                eos_token_id=SAFE_EOS_ID,
            )
            new_tokens = gen[0, input_ids.shape[1]:]
            text = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)
            outputs.append({
                "id": rec.get("id", i),
                "query_vi": rec["query_vi"],
                "type": rec.get("type", "unknown"),
                "model_output": postprocess_output(text),
            })

    save_json(outputs, output_path)
    print("Wrote:", output_path)
    print("Minutes:", round((time.time() - start_all) / 60, 2))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return outputs

In [12]:
# 12. Sinh output validation
RUN_VALIDATION = True
MODEL_FOR_INFERENCE = OUTPUT_DIR if OUTPUT_DIR.exists() else MODEL_DIR

if RUN_VALIDATION and valid_records:
    valid_outputs = generate_predictions(MODEL_FOR_INFERENCE, valid_records, VALID_OUTPUT_PATH, name="validation")
    print("\nOutput mẫu:")
    print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:1200])
else:
    valid_outputs = []
    print("Skip validation generation")

Load for generation: /kaggle/working/gpt2_math_baseline_ckpt | device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

validation:   0%|          | 0/1000 [00:00<?, ?it/s]

Wrote: /kaggle/working/valid_output.json
Minutes: 19.3

Output mẫu:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "Susan đang chơi một ván cờ bàn có 24 ô từ ô đầu tiên đến ô cuối cùng, vậy cô ấy cần tổng cộng 24 * 12 = 240 ô từ ô cuối cùng đến ô cuối. Cô ấy cũng đang chơi một trận đấu cờ bàn có 6 ô từ ô thứ hai đến ô cuối, vậy cô cần tổng cộng 6 * 6 = 240 ô. Vì vậy, Susan cần 240 ô để giành chiến thắng. Đáp án là: 240"
}


In [13]:
# 13. Đánh giá validation và in vài case để đọc lỗi
CASES_TO_SHOW = 8


def align_by_id(preds, golds):
    if all("id" in x for x in preds) and all("id" in x for x in golds):
        pred_map = {str(x["id"]): x for x in preds}
        return [(pred_map[str(g["id"])], g) for g in golds if str(g["id"]) in pred_map]
    return list(zip(preds, golds))


def evaluate_predictions(preds, golds):
    rows = []
    for row_index, (pred, gold) in enumerate(align_by_id(preds, golds)):
        gold_answer = extract_answer_text(gold.get("response_vi"), allow_last_number=True)
        pred_answer = extract_answer_text(pred.get("model_output"), allow_last_number=False)
        gold_num = parse_number(gold_answer)
        pred_num = parse_number(pred_answer)
        rel_err = relative_error(pred_num, gold_num)
        score = score_one(rel_err, pred_answer is not None)
        rows.append({
            "row_index": row_index,
            "id": gold.get("id"),
            "type": gold.get("type"),
            "query_vi": gold.get("query_vi"),
            "model_output": pred.get("model_output"),
            "gold_answer": gold_answer,
            "pred_answer": pred_answer,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": rel_err,
            "extractable": pred_answer is not None,
            "score": score,
        })
    return rows


def score_summary(rows):
    n = len(rows)
    raw = sum(r["score"] for r in rows)
    return {
        "n": n,
        "raw_score": raw,
        "max_raw_score": 10 * n,
        "score_10": raw / n if n else 0,
        "extractable_rate": sum(r["extractable"] for r in rows) / n if n else 0,
        "buckets": {str(s): sum(r["score"] == s for r in rows) for s in [10, 5, 1, 0]},
    }


def show_cases(title, rows):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if not rows:
        print("Không có case")
        return
    cols = ["row_index", "id", "type", "score", "rel_error", "gold_answer", "pred_answer", "query_vi", "model_output"]
    display(pd.DataFrame(rows[:CASES_TO_SHOW])[cols])


if valid_outputs:
    eval_rows = evaluate_predictions(valid_outputs, valid_records)
    summary = score_summary(eval_rows)
    print("Validation summary:")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    by_type = pd.DataFrame(eval_rows).groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()
    display(by_type[["type", "n", "score_10", "extractable_rate", "raw_score", "max_raw_score", "buckets"]])

    show_cases("Một vài case đúng", [r for r in eval_rows if r["score"] == 10])
    show_cases("Một vài case sai", [r for r in eval_rows if r["score"] == 0])
    show_cases("Một vài case không tách được đáp án", [r for r in eval_rows if not r["extractable"]])
else:
    eval_rows = []
    summary = None
    print("Không có validation output để đánh giá")

Validation summary:
{
  "n": 1000,
  "raw_score": 650,
  "max_raw_score": 10000,
  "score_10": 0.65,
  "extractable_rate": 0.84,
  "buckets": {
    "10": 40,
    "5": 19,
    "1": 155,
    "0": 786
  }
}


/tmp/ipykernel_22/1188216732.py:68: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_type = pd.DataFrame(eval_rows).groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()


,type,n,score_10,extractable_rate,raw_score,max_raw_score,buckets
0,GSM_AnsAug,209,0.444976,0.952153,93,2090,"{'10': 3, '5': 4, '1': 43, '0': 159}"
1,GSM_FOBAR,122,0.795082,0.655738,97,1220,"{'10': 8, '5': 0, '1': 17, '0': 97}"
2,GSM_Rephrased,197,0.619289,1.000000,122,1970,"{'10': 3, '5': 9, '1': 47, '0': 138}"
3,GSM_SV,97,0.268041,0.319588,26,970,"{'10': 2, '5': 0, '1': 6, '0': 89}"
4,MATH_AnsAug,173,0.780347,0.884393,135,1730,"{'10': 10, '5': 3, '1': 20, '0': 140}"
5,MATH_FOBAR,45,1.044444,0.777778,47,450,"{'10': 4, '5': 0, '1': 7, '0': 34}"
6,MATH_Rephrased,116,0.525862,0.965517,61,1160,"{'10': 5, '5': 0, '1': 11, '0': 100}"
7,MATH_SV,41,1.682927,0.804878,69,410,"{'10': 5, '5': 3, '1': 4, '0': 29}"



Một vài case đúng


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,26,26,MATH_AnsAug,10,0.0,4,4,"Giả sử $a$ thay đổi nghịch đảo với $b^2$. Nếu $a=9$ khi $b=2$, hãy tìm giá trị của $a$ khi $b=3$.","Chúng ta có thể viết lại phương trình dưới dạng $(a+b)^2$. Để tìm giá trị tối đa của $a$, chúng ta có thể nhân cả hai vế của phương trình với $b$. Vì $a=3$, chúng ta cần tìm gi..."
1,32,32,GSM_AnsAug,10,0.0,45,45,"Trong nhà hàng, một chiếc burger có giá 9 USD và một chiếc pizza có giá gấp đôi. Một chiếc bánh pizza và ba chiếc bánh mì kẹp thịt sẽ có giá bao nhiêu?","Một chiếc burger có thể có giá 9 đô la, vì vậy một chiếc burger sẽ có giá 9 * $9 = $20. Một chiếc pizza có thể có ba chiếc bánh pizza, vì vậy ba chiếc bánh sẽ có giá 3 * $20 = ..."
2,37,37,MATH_AnsAug,10,0.0,\frac{1}{4},\frac{1},Bán kính của một hình cầu là đơn vị $p$ và bán kính của một bán cầu là đơn vị $2p$. Tỉ số giữa thể tích của hình cầu và thể tích của bán cầu là bao nhiêu?,"Thể tích của hình tròn là $\frac{p}{2}(2p)$. Thể tích của bán kính là $\frac{\pi}{2} = \frac{2}{2}$. Đơn giản hóa, chúng ta có $\frac{2p}{2p} = \boxed{\frac{1}{2}}$. Đáp án là:..."
3,62,62,MATH_SV,10,0.0,6,6,"Chỉ sử dụng mỗi chữ số trong số năm chữ số $2, 4, X, 7$ và $9$ một lần để tạo thành số nguyên có ba chữ số và số nguyên có hai chữ số sẽ được nhân với nhau. Số nguyên có ba chữ...","Để giải bài toán này, chúng ta cần xác định giá trị của x, biểu thị số nguyên có 3 chữ số. Chúng ta biết rằng chỉ sử dụng mỗi số trong số 5 chữ số, vì vậy chúng ta có thể viết ..."
4,67,67,GSM_Rephrased,10,0.0,60,60,"Nếu Emma có thể hoàn thành việc chạy vòng quanh hai đấu trường trong 20 giờ và Fernando phải mất gấp đôi thời gian để chạy hết chặng đường của mình, thì tổng thời gian để cả ha...",Nếu Emma có thể chạy vòng quanh 2 đấu trường trong 10 phút thì cô ấy có thể chạy được 20 + 10 = 30 phút. Nếu Fernando chạy vòng quanh 20 phút thì anh ấy có thể đi được 30 - 20 ...
5,68,68,MATH_AnsAug,10,0.0,10,10,"Một người bán tạp hóa trưng bày các lon, trong đó hàng trên cùng có một lon và mỗi hàng dưới có nhiều hơn hàng phía trên 2 lon. Nếu màn hình chứa 100 lon thì nó chứa bao nhiêu ...","Nếu màn hình chứa 200 lon thì hàng đầu tiên có 200/200 = 10 lon. Hàng thứ hai có 100/100 = 10 lon nên hàng thứ nhất có 100/10 = 10 lon, hàng thứ hai có 10/10 = 5 lon. Tổng cộng..."
6,99,99,MATH_AnsAug,10,0.0,2,2,Tìm giá trị lớn nhất của \[f(x) = 3x - x^3\]với $0 \le x \le \sqrt{3}.$,Chúng ta có thể viết lại phương trình dưới dạng $f(x)$ bằng cách sử dụng công thức bậc hai. Chúng ta có thể bắt đầu bằng cách sử dung tích của $f(0) = \frac{3}{2}$. Để tìm giá ...
7,156,156,MATH_AnsAug,10,0.0,1,1,"Cho $a,$ $b,$ $c$ là các số phức phân biệt sao cho \[\frac{a}{1 - b} = \frac{b}{1 - c} = \frac{c}{1 - a} = k.\]Tìm tổng tất cả các giá trị có thể có của $k.$","Chúng ta có thể viết lại phương trình dưới dạng $k = \frac{\begin{align*} a}{1 - 2a} = k$. Thay thế các giá trị đã cho, chúng ta có $k = 1 - 2a = \frac {1}{2} = \boxed{1}$. Đáp..."



Một vài case sai


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,0,0,GSM_Rephrased,0,5.486486,37,240,"Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nh...","Susan đang chơi một ván cờ bàn có 24 ô từ ô đầu tiên đến ô cuối cùng, vậy cô ấy cần tổng cộng 24 * 12 = 240 ô từ ô cuối cùng đến ô cuối. Cô ấy cũng đang chơi một trận đấu cờ bà..."
1,1,1,MATH_Rephrased,0,0.736842,19,5,"Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?","Chu vi của tam giac PQR là $\angle PRQ = \angle PrQ$, trong đó $PRQ$ là độ dài của một tam giác. Vì vậy, chu vi của một tam giac PQRP là $\angle PrQ = \boxed{5}$. Đáp án là: 5"
2,3,3,GSM_Rephrased,0,0.641026,39,14,"Sau khi bắt đầu với 50 quả bóng bay, Claire đưa cho một bé gái 1 quả bóng bay, mất 12 quả bóng bay, cho thêm 9 quả bóng nữa và lấy 11 quả bóng bay từ đồng nghiệp của mình. Hiện...","Claire đưa cho một em gái 1 quả bong bóng bay, vì vậy cô ấy đưa cho cô ấy 1 quả bóng. Cô ấy đã mất 12 quả bong bóng, vậy cô ấy đã mất tổng cộng 12 + 12 = 18 quả bong bóng. Cô b..."
3,5,5,GSM_AnsAug,0,5.666667,90,600,"Hans đặt phòng ở khách sạn. Khách sạn có 10 tầng, mỗi tầng có 10 phòng giống nhau. Do xảy ra tai nạn nên tầng cuối cùng không còn chỗ cho khách. Xem xét không có khách nào khác...","Tầng đầu tiên có 10 phòng, mỗi phòng có 10 phòng nên có 10 * 10 = 50 phòng. Tầng thứ hai có 10 phòng và mỗi phòng có 20 phòng nên có 50 * 20 = 300 phòng. Tầng cuối cùng có 10 p..."
4,6,6,GSM_SV,0,NaN,12,None,Bà Dunbar đang cắm hoa cho đám cưới của cháu gái bà. Cô ấy cần làm 5 bó hoa và 7 món đồ trang trí bàn ăn. Cô sử dụng x bông hồng trắng để trang trí mỗi bàn và 5 bông hồng trắng...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số lượng bông hồng trắng mà bà Dunbar cần làm cho mỗi bàn ăn. Chúng ta hãy chia nhỏ thông tin đã cho..."
5,7,7,GSM_FOBAR,0,NaN,9,None,"Grace bắt đầu công việc kinh doanh cảnh quan của riêng mình. Cô tính phí 6 đô la một giờ cho việc cắt cỏ, 11 đô la cho việc nhổ cỏ và x đô la cho việc phủ lớp phủ. Vào tháng 9,...","Grace bắt đầu công tác kinh doanh của mình với mức lương 6 đô la mỗi giờ. Cô tính toán phí cắt cỏ, nhổ cỏ và phủ lớp sơn. Cô ấy tính phí 6 USD mỗi giờ, vậy tổng chi phí của cô ..."
6,8,8,MATH_Rephrased,0,0.866667,60,8,"Xác định giá trị cao nhất trong số các bội số chung nhỏ nhất của 12 và 2, 12 và 4, 12 và 6, 12 và 8, 12 và 10, 12 và 12. Hãy thể hiện câu trả lời của bạn dưới dạng số nguyên.","Chúng ta có thể tìm thấy các bội số của 12 và hai, 12 và 2 bằng cách sử dụng công thức: $12 + 2 = \boxed{8}$. Đáp án là: 8"
7,9,9,GSM_AnsAug,0,11.000000,5,60,"Bob được hỗ trợ tiền thuê nhà vì anh ấy có thu nhập thấp. Nếu anh ta được tăng lương 0,50 USD/giờ và làm việc 40 giờ một tuần, anh ta sẽ thực sự kiếm được bao nhiêu tiền một tu...","Bob được tăng lương từ 0,50 đến 0,50 đô la mỗi giờ, vì vậy anh ấy có thể kiếm được 60 đô la x 0,50 = $60. Anh ta cũng được tăng lương gấp đôi từ 0,5 đến 1 đô la mỗi tuần, vì vậ..."



Một vài case không tách được đáp án


,row_index,id,type,score,rel_error,gold_answer,pred_answer,query_vi,model_output
0,6,6,GSM_SV,0,None,12,None,Bà Dunbar đang cắm hoa cho đám cưới của cháu gái bà. Cô ấy cần làm 5 bó hoa và 7 món đồ trang trí bàn ăn. Cô sử dụng x bông hồng trắng để trang trí mỗi bàn và 5 bông hồng trắng...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số lượng bông hồng trắng mà bà Dunbar cần làm cho mỗi bàn ăn. Chúng ta hãy chia nhỏ thông tin đã cho..."
1,7,7,GSM_FOBAR,0,None,9,None,"Grace bắt đầu công việc kinh doanh cảnh quan của riêng mình. Cô tính phí 6 đô la một giờ cho việc cắt cỏ, 11 đô la cho việc nhổ cỏ và x đô la cho việc phủ lớp phủ. Vào tháng 9,...","Grace bắt đầu công tác kinh doanh của mình với mức lương 6 đô la mỗi giờ. Cô tính toán phí cắt cỏ, nhổ cỏ và phủ lớp sơn. Cô ấy tính phí 6 USD mỗi giờ, vậy tổng chi phí của cô ..."
2,25,25,MATH_Rephrased,0,None,-2,None,Xác định giá trị nhỏ nhất của y thỏa mãn phương trình $3y^2 + 5y + 2 = 4$.,"Chúng ta có thể viết lại phương trình dưới dạng $3y + 2y + 2=4$. Vì vậy, chúng ta có thể thay thế các giá trị đã cho vào phương trình để tìm giá trị nhỏ hơn của $3y$ thỏa mãn p..."
3,27,27,GSM_FOBAR,0,None,2,None,"Annie thích ăn bánh quy. Cô ấy ăn 5 cái bánh quy vào thứ Hai, gấp x lần vào thứ Ba và vào thứ Tư nhiều hơn 40% so với thứ Ba. Annie đã ăn bao nhiêu cái bánh quy trong ba ngày n...","Vào thứ Hai, Annie đã ăn 5 cái. Vào thứ Ba, cô ấy ăn gấp đôi, tức là 5 * x = 20x. Vào thứ Tư, cô ấy đã ăn gấp 3 lần, tức là 3 * x = 10x. Vào Thứ Tư, cô ăn gấp 2 lần, tức gấp 3 ..."
4,29,29,GSM_FOBAR,0,None,10,None,"Henry muốn mua một chiếc áo phông, một chiếc quần jeans và một đôi tất. Chiếc quần jean có giá gấp đôi chiếc áo phông và chiếc áo phông đắt hơn đôi tất x $. Những chiếc tất có ...","Henry muốn mua một cái quần jean, một chiếc áo khoác và một đôi giày. Chiếc quần jeans có giá 5 đô la, vì vậy anh ấy sẽ trả 5 * $5 = $10 cho chiếc quần jean. Chiếc áo phông có ..."
5,45,45,MATH_SV,0,None,225,None,Xanthia có thể đọc 100 trang mỗi giờ và Molly có thể đọc 50 trang mỗi giờ. Nếu mỗi người đọc cùng một cuốn sách và cuốn sách có X trang thì Molly sẽ mất nhiều hơn Xanthia 135 p...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số trang Molly sẽ đọc trong một ngày. Chúng ta hãy chia nhỏ thông tin đã cho: Số trang Molly đọc: 10..."
6,52,52,GSM_SV,0,None,25,None,Debby và Maggie đồng ý chia nhau một khoản tiền để Debby nhận x% và Maggie nhận phần còn lại. Nếu phần của Maggie là 4.500 USD thì họ chia nhau 6000. Giá trị của biến x chưa bi...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số tiền Debby nhận được từ Maggie. Chúng ta biết rằng số tiền Debbie nhận được là 4.000 USD. Số tiền..."
7,57,57,GSM_SV,0,None,60,None,Darnell trả 12 USD cho việc nhắn tin và gọi điện không giới hạn trên điện thoại của mình mỗi tháng. Một gói điện thoại thay thế sẽ tính phí 1 USD cho mỗi 30 tin nhắn và 3 USD c...,"Để giải quyết vấn đề này, chúng ta cần xác định giá trị của x, đại diện cho số phút anh ấy nhắn tin mỗi tháng. Hãy chia nhỏ thông tin đã cho: Số phút anh Darnell nhắn tin mỗi n..."


In [14]:
# 14. Sinh test_predictions.json cho Phase 2
RUN_TEST_INFERENCE = True

if RUN_TEST_INFERENCE and test_records:
    test_outputs = generate_predictions(MODEL_FOR_INFERENCE, test_records, TEST_OUTPUT_PATH, name="test")
    print("\nTest output mẫu:")
    print(json.dumps(test_outputs[:2], ensure_ascii=False, indent=2)[:1200])
else:
    print("Không có test.json, bỏ qua bước test inference")

Không có test.json, bỏ qua bước test inference


In [15]:
# 15. Kiểm tra file đầu ra chính
for p in [OUTPUT_DIR, VALID_OUTPUT_PATH, TEST_OUTPUT_PATH]:
    p = Path(p)
    if p.exists():
        size = p.stat().st_size if p.is_file() else "<dir>"
        print(p, "|", size)

print("\nDone.")

/kaggle/working/gpt2_math_baseline_ckpt | <dir>
/kaggle/working/valid_output.json | 815248

Done.
